# Participant-Level Data Splitting Strategy

This notebook implements the participant-level data splitting strategy for the PADS movement metadata dataset. The objective is to prepare the dataset for machine learning by dividing participants into training, validation, and testing sets while preventing participant-level data leakage.

The splitting strategy ensures that all recordings belonging to the same participant remain within a single dataset, allowing unbiased model training and evaluation.

In [101]:
# ============================================================================
# Import Libraries
# ============================================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

# Set random seed
np.random.seed(42)

In [102]:
# ============================================================
# Load  Datasets
# ============================================================

patients_df = pd.read_csv("../data/processed/patients.csv")
movement_df = pd.read_csv("../data/processed/movement_metadata.csv")

print(f"Patients: {len(patients_df)}")
print(f"Movement records: {len(movement_df)}")

display(patients_df.head())
display(movement_df.head())

Patients: 469
Movement records: 5159


,patient_id,study_id,condition,label,disease_comment,age_at_diagnosis,age,height_cm,weight_kg,gender,handedness,appearance_in_kinship,appearance_in_first_grade_kinship,effect_of_alcohol_on_tremor
0,1,PADS,Healthy,0,-,56,56,173,78,male,right,True,True,Unknown
1,2,PADS,Other Movement Disorders,2,Left-Sided resting tremor and hypokinesia with...,69,81,193,104,male,right,False,NaN,No effect
2,3,PADS,Healthy,0,-,45,45,170,78,female,right,False,NaN,Unknown
3,4,PADS,Parkinson's,1,IPS akinetic-rigid type,63,67,161,90,female,right,False,NaN,No effect
4,5,PADS,Parkinson's,1,IPS tremordominant type,65,75,172,86,male,left,False,NaN,Unknown


,patient_id,device,sampling_rate,task,samples,left_file,right_file
0,1,Apple Watch Series 4,100,CrossArms,1024,timeseries/001_CrossArms_LeftWrist.txt,timeseries/001_CrossArms_RightWrist.txt
1,1,Apple Watch Series 4,100,DrinkGlas,1024,timeseries/001_DrinkGlas_LeftWrist.txt,timeseries/001_DrinkGlas_RightWrist.txt
2,1,Apple Watch Series 4,100,Entrainment,2048,timeseries/001_Entrainment_LeftWrist.txt,timeseries/001_Entrainment_RightWrist.txt
3,1,Apple Watch Series 4,100,HoldWeight,1024,timeseries/001_HoldWeight_LeftWrist.txt,timeseries/001_HoldWeight_RightWrist.txt
4,1,Apple Watch Series 4,100,LiftHold,1024,timeseries/001_LiftHold_LeftWrist.txt,timeseries/001_LiftHold_RightWrist.txt


In [103]:
# =============================================================================
# Dataset Summary
# =============================================================================

print(f"Movement recordings : {len(movement_df):,}")
print(f"Participants        : {movement_df['patient_id'].nunique():,}")
print(f"Motor tasks         : {movement_df['task'].nunique():,}")

Movement recordings : 5,159
Participants        : 469
Motor tasks         : 11


In [104]:
# =============================================================================
# Dataset Structure
# =============================================================================

movement_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5159 entries, 0 to 5158
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   patient_id     5159 non-null   int64 
 1   device         5159 non-null   object
 2   sampling_rate  5159 non-null   int64 
 3   task           5159 non-null   object
 4   samples        5159 non-null   int64 
 5   left_file      5159 non-null   object
 6   right_file     5159 non-null   object
dtypes: int64(3), object(4)
memory usage: 282.3+ KB


In [105]:
# =============================================================================
# Check Unique Participants
# =============================================================================

movement_df["patient_id"].nunique()

469

In [106]:
# ============================================================
# Split Patients into Training and Temporary Sets
# ============================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    gss.split(
        patients_df,
        groups=patients_df["patient_id"]
    )
)

train_patients = patients_df.iloc[train_idx].reset_index(drop=True)
temp_patients = patients_df.iloc[temp_idx].reset_index(drop=True)

In [107]:
# ============================================================
# Verify First Split
# ============================================================

print(f"Training patients : {len(train_patients):,}")
print(f"Temporary patients: {len(temp_patients):,}")

print()

print(f"Unique training patients : {train_patients['patient_id'].nunique()}")
print(f"Unique temporary patients: {temp_patients['patient_id'].nunique()}")

Training patients : 328
Temporary patients: 141

Unique training patients : 328
Unique temporary patients: 141


In [108]:
# ============================================================
# Split Temporary Patients into Validation and Test Sets
# ============================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    gss.split(
        temp_patients,
        groups=temp_patients["patient_id"]
    )
)

validation_patients = temp_patients.iloc[val_idx].reset_index(drop=True)
test_patients = temp_patients.iloc[test_idx].reset_index(drop=True)

In [109]:
# ============================================================
# Check HC / PD / OMD Distribution
# ============================================================

def map_group(condition):
    if condition == "Healthy":
        return "HC"
    elif condition == "Parkinson's":
        return "PD"
    else:
        return "OMD"

for name, df in [
    ("Training", train_patients),
    ("Validation", validation_patients),
    ("Test", test_patients),
]:

    temp = df.copy()
    temp["Group"] = temp["condition"].apply(map_group)

    counts = temp["Group"].value_counts().reindex(["HC", "PD", "OMD"], fill_value=0)
    percentages = (
        temp["Group"]
        .value_counts(normalize=True)
        .reindex(["HC", "PD", "OMD"], fill_value=0)
        * 100
    ).round(2)

    print(f"\n{name} Set")
    print("=" * 40)

    summary = pd.DataFrame({
        "Count": counts,
        "Percentage (%)": percentages
    })

    print(summary)


Training Set
       Count  Percentage (%)
Group                       
HC        55           16.77
PD       189           57.62
OMD       84           25.61

Validation Set
       Count  Percentage (%)
Group                       
HC        13           18.57
PD        45           64.29
OMD       12           17.14

Test Set
       Count  Percentage (%)
Group                       
HC        11           15.49
PD        42           59.15
OMD       18           25.35


### Class Distribution Summary

The HC, PD, and OMD groups are represented across the training, validation, and test datasets. Although minor differences exist due to the relatively small validation and test sets, the overall class distribution remains reasonably consistent, indicating that the participant-level splitting strategy preserved the class.

In [110]:
# =============================================================================
# Check for Data Leakage
# =============================================================================

train_ids = set(train_patients["patient_id"])
validation_ids = set(validation_patients["patient_id"])
test_ids = set(test_patients["patient_id"])

print("Train ∩ Validation:", len(train_ids & validation_ids))
print("Train ∩ Test      :", len(train_ids & test_ids))
print("Validation ∩ Test :", len(validation_ids & test_ids))

Train ∩ Validation: 0
Train ∩ Test      : 0
Validation ∩ Test : 0


### Data Leakage Summary
A participant-level data splitting strategy was successfully implemented using GroupShuffleSplit. The movement metadata dataset was divided into training, validation, and testing datasets while preserving participant independence. Verification confirmed that there was no overlap of participant IDs across the three datasets, eliminating participant-level data leakage. The resulting datasets are now prepared for feature engineering, model development, and performance evaluation.

In [111]:
# ============================================================
#  Participant Split to Movement Dataset
# ============================================================

train_df = movement_df[
    movement_df["patient_id"].isin(train_patients["patient_id"])
].reset_index(drop=True)

validation_df = movement_df[
    movement_df["patient_id"].isin(validation_patients["patient_id"])
].reset_index(drop=True)

test_df = movement_df[
    movement_df["patient_id"].isin(test_patients["patient_id"])
].reset_index(drop=True)

In [112]:
# =============================================================================
# Verify Final Split
# =============================================================================

print(f"Training participants   : {train_df['patient_id'].nunique()}")
print(f"Validation participants : {validation_df['patient_id'].nunique()}")
print(f"Test participants       : {test_df['patient_id'].nunique()}")

print()

print(f"Training recordings   : {len(train_df):,}")
print(f"Validation recordings : {len(validation_df):,}")
print(f"Test recordings       : {len(test_df):,}")

Training participants   : 328
Validation participants : 70
Test participants       : 71

Training recordings   : 3,608
Validation recordings : 770
Test recordings       : 781


In [113]:
# ============================================================================
# Save Split Datasets
# ============================================================================

OUTPUT_PATH = Path("../data/processed")

train_df.to_csv(OUTPUT_PATH / "train_metadata.csv", index=False)
validation_df.to_csv(OUTPUT_PATH / "validation_metadata.csv", index=False)
test_df.to_csv(OUTPUT_PATH / "test_metadata.csv", index=False)

print("Datasets saved successfully!\n")

print(f"Train metadata saved      : {len(train_df):,} records")
print(f"Validation metadata saved : {len(validation_df):,} records")
print(f"Test metadata saved       : {len(test_df):,} records")

Datasets saved successfully!

Train metadata saved      : 3,608 records
Validation metadata saved : 770 records
Test metadata saved       : 781 records


### Conclusion

A participant-level data splitting strategy was successfully implemented using GroupShuffleSplit. Participant assignments were first generated from the patient metadata and then applied to the movement metadata, ensuring that all recordings belonging to the same participant remained within a single dataset. Final verification confirmed that no participant appeared in more than one dataset, eliminating participant-level data leakage and making the resulting datasets suitable for feature engineering, model development, and performance evaluation.